In [0]:
import importlib
from config.config import *

from pyspark.sql.functions import *
from pyspark.sql.types import *
import re

In [0]:
df = spark.read.format("delta").load(
    f"{bronze_path}/bronze_estat_tour_occ_mnor_raw"
)
display(df.head(10))

In [0]:
header_row = df.filter(
    col("_c0").startswith("freq,")
).first()

period_columns = []

for c in df.columns[1:]:
    period = str(header_row[c]).strip()

    if re.fullmatch(r"\d{4}-\d{2}", period):
        period_columns.append((c, period))

print("Monthly columns:", len(period_columns))

In [0]:
df = df.filter(
    ~col("_c0").startswith("freq,")
)

df.display()

In [0]:
dimension_names = header_row["_c0"].split(",")

# Remove TIME_PERIOD suffix from geo
dimension_names[-1] = re.sub(
    r"\\TIME_PERIOD$",
    "",
    dimension_names[-1]
).strip()

period_columns = []

for c in df.columns[1:]:
    period = str(header_row[c]).strip()

    if re.fullmatch(r"\d{4}-\d{2}", period):
        period_columns.append((c, period))

print("Dimensions:", dimension_names)
print("Monthly columns:", len(period_columns))

In [0]:
df = df.filter(
    ~col("_c0").startswith("freq,")
)

In [0]:
# Split dimensions
df = df.withColumn(
    "dimensions",
    split(col("_c0"), ",")
)

df = df.select(
    trim(col("dimensions")[0]).alias("frequency"),
    trim(col("dimensions")[1]).alias("accommodation_unit"),
    trim(col("dimensions")[2]).alias("unit_code"),
    trim(col("dimensions")[3]).alias("geo_code"),
    *[col(c) for c, _ in period_columns]
)

In [0]:
observations = array(*[
    struct(
        lit(period).alias("period"),
        col(c).alias("raw_observation")
    )
    for c, period in period_columns
])

df = df.withColumn(
    "observations",
    explode(observations)
)

df = df.select(
    "frequency",
    "accommodation_unit",
    "unit_code",
    "geo_code",
    "observations.period",
    "observations.raw_observation"
)

In [0]:
df = df.withColumn(
    "raw_observation",
    trim(col("raw_observation"))
)

# Extract numerical value
df = df.withColumn(
    "numeric_string",
    regexp_extract(
        col("raw_observation"),
        r"^([+-]?(?:\d+(?:\.\d*)?|\.\d+))",
        1
    )
)

df = df.withColumn(
    "occupancy_rate",
    expr("try_cast(numeric_string AS DOUBLE)")
)

# Extract observation flags
df = df.withColumn(
    "observation_flag",
    regexp_extract(
        col("raw_observation"),
        r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)\s*([a-zA-Z]+)$",
        1
    )
)

df = df.drop("numeric_string")

In [0]:
df = df.withColumn(
    "period",
    to_date(
        concat(col("period"), lit("-01")),
        "yyyy-MM-dd"
    )
)

In [0]:
df = df.withColumn(
    "geography_type",
    when(
        col("geo_code").startswith("EU"),
        "eu_aggregate"
    ).otherwise("country")
)

In [0]:
silver_df = df.select(
    "frequency",
    "accommodation_unit",
    "unit_code",
    "geo_code",
    "geography_type",
    "period",
    "occupancy_rate",
    "observation_flag"
)

display(silver_df)

In [0]:
output_path = f"{silver_path}/estat_tour_occ_mnor"

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(output_path)

print("Data cleaning for estat_tour_occ_mnor completed")

In [0]:
spark.read.format("delta").load(output_path).display()